In [2]:
%pip install torch transformers pandas tqdm

     ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/114.5 MB 1.3 MB/s eta 0:01:30
     ---------------------------------------- 0.1/114.5 MB 1.1 MB/s eta 0:01:49
     ---------------------------------------- 0.2/114.5 MB 1.5 MB/s eta 0:01:15
     ---------------------------------------- 0.4/114.5 MB 2.2 MB/s eta 0:00:53
     ---------------------------------------- 0.8/114.5 MB 3.2 MB/s eta 0:00:36
     ---------------------------------------- 1.3/114.5 MB 4.7 MB/s eta 0:00:25
      --------------------------------------- 2.4/114.5 MB 7.1 MB/s eta 0:00:16
     - ------------------------------------- 4.2/114.5 MB 11.1 MB/s eta 0:00:10
     -- ------------------------------------ 7.1/114.5 MB 16.7 MB/s eta 0:00:07
     --- ----------------------------------- 9.9/114.5 MB 21.1 MB/s eta 0:00:05
     ---- --------------------------------- 12.8/114.5 MB 59.5 MB/s eta 0:00:02
     ---- --------------------------------- 14.


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
%pip install torch torchvision torchaudio --force-reinstall

Note: you may need to restart the kernel to use updated packages.Collecting torch
  Using cached torch-2.11.0-cp310-cp310-win_amd64.whl (114.5 MB)
     ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
      --------------------------------------- 0.1/3.7 MB 1.1 MB/s eta 0:00:04
     - -------------------------------------- 0.2/3.7 MB 1.7 MB/s eta 0:00:03
     ----- ---------------------------------- 0.5/3.7 MB 3.3 MB/s eta 0:00:01
     ---------- ----------------------------- 1.0/3.7 MB 5.5 MB/s eta 0:00:01
     ---------------------- ----------------- 2.1/3.7 MB 8.8 MB/s eta 0:00:01
     ---------------------------------------- 3.7/3.7 MB 13.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/328.7 kB ? eta -:--:--
     ---------------------------------------- 328.7/328.7 kB ? eta 0:00:00
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
  Using cached filelock-3.29.0-py3-none-any.whl (39 kB)
     ---------------------------------------- 0.0/1.1


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
%pip install mkl mkl-include

     ---------------------------------------- 0.0/180.8 MB ? eta -:--:--
     -------------------------------------- 0.0/180.8 MB 653.6 kB/s eta 0:04:37
     ---------------------------------------- 0.2/180.8 MB 1.5 MB/s eta 0:01:59
     ---------------------------------------- 0.4/180.8 MB 2.7 MB/s eta 0:01:07
     ---------------------------------------- 0.9/180.8 MB 4.9 MB/s eta 0:00:37
     ---------------------------------------- 1.8/180.8 MB 8.3 MB/s eta 0:00:22
      -------------------------------------- 3.3/180.8 MB 12.6 MB/s eta 0:00:15
     - ------------------------------------- 6.1/180.8 MB 19.6 MB/s eta 0:00:09
     -- ----------------------------------- 10.1/180.8 MB 28.1 MB/s eta 0:00:07
     --- --------------------------------- 15.4/180.8 MB 108.8 MB/s eta 0:00:02
     ---- -------------------------------- 21.3/180.8 MB 131.2 MB/s eta 0:00:02
     ----- ------------------------------- 26.4/180.8 MB 131.2 MB/s eta 0:00:02
     ------ ------------------------------ 32.4


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
# Thay Admin bằng tên user của bạn
os.environ["Admin"]="TRUE"
import torch
print(torch.__version__)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [7]:
import json
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

def extract_modernbert_embeddings(json_filepath, output_csv):
    # 1. Khởi tạo Tokenizer và Model ModernBERT
    # Đây là mô hình mới (2024/2025) tối ưu cho việc xử lý văn bản dài và hiệu suất cao
    model_name = "answerdotai/ModernBERT-base"
    print(f"--- Đang tải mô hình {model_name} ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    # Kiểm tra và sử dụng GPU nếu có
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval() # Chuyển sang chế độ inference
    print(f"Mô hình đang chạy trên: {device}")

    # 2. Đọc dữ liệu JSON
    with open(json_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    results = []
    
    # 3. Vòng lặp trích xuất
    print("--- Bắt đầu trích xuất Embedding 768D ---")
    for item_id, item_data in tqdm(data.items(), desc="Processing"):
        content = item_data.get("content", "")
        analysis = item_data.get("analysis", "")
        
        # Tạo chuỗi kết hợp Đề bài và Lời giải để BERT học trọn vẹn logic bài toán
        # [SEP] là token phân tách tiêu chuẩn trong BERT
        combined_text = f"Đề bài: {content} [SEP] Lời giải: {analysis}"
        
        # Tokenize văn bản, giới hạn 512 tokens (giới hạn chuẩn của BERT-base)
        inputs = tokenizer(
            combined_text, 
            return_tensors="pt", 
            truncation=True, 
            max_length=512, 
            padding=True
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        # Lấy vector [CLS] - đây là vector đại diện cho toàn bộ ngữ nghĩa câu (768 chiều)
        # Chúng ta lấy ở vị trí index 0 của layer cuối cùng
        cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        
        # Chuẩn bị dữ liệu dòng
        row = {"ID": item_id}
        for i, val in enumerate(cls_embedding):
            row[f"BERT_{i}"] = float(val)
            
        results.append(row)

    # 4. Lưu ra file CSV
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\n✅ Hoàn tất! Đã lưu {len(df)} vector vào file: {output_csv}")

if __name__ == "__main__":
    # Cấu hình đường dẫn file của bạn
    INPUT_JSON = "D:/IntelligentTesting/IntelligentTesting/XES3G5M/XES3G5M/metadata/question_full.json"
    OUTPUT_CSV = "bert_embeddings_768d.csv"
    
    extract_modernbert_embeddings(INPUT_JSON, OUTPUT_CSV)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.